# ED5J990H5VAZT

Packages

In [ ]:
# Imports
from foodcast.imports import *
os.chdir(find_project_root())
BASE_DIR, DATA_DIR_1, DATA_DIR_2, DATA_DIR_3, DATA_DIR_3_x = return_dir()
DATA_DIR_3_1, DATA_DIR_3_2, DATA_DIR_3_3, _, _, _ = DATA_DIR_3_x

# Settings
notebook_settings()  

# Load Data
location_ids_by_coverage = load_loc_ids()
locations, before_after_details_true, items_tagged, customers = load_static()
loc_id = 'ED5J990H5VAZT'
df_uncleaned = load_single_restaurant(loc_id)

In [ ]:
# Load remapping data from YAML (like loc2 does)
labeling_path = Path('scripts') / 'labeling'
remapping_path = labeling_path / 'remapping' / 'loc5_remappings.yaml'
with open(remapping_path, "r", encoding="utf-8") as f:
    remapping = yaml.load(f, Loader=yaml.FullLoader)

# Extract modification patterns from YAML
MODS_VEGAN_MEAT = remapping['modification_patterns']['MODS_VEGAN_MEAT']
MODS_MEAT = remapping['modification_patterns']['MODS_MEAT']
MODS_VEGAN_DAIRY_EGG = remapping['modification_patterns']['MODS_VEGAN_DAIRY_EGG']
MODS_DAIRY_EGG = remapping['modification_patterns']['MODS_DAIRY_EGG']
MODS_EXPLICIT_VEGAN = remapping['modification_patterns']['MODS_EXPLICIT_VEGAN']
MODS_NONE = remapping['modification_patterns']['MODS_NONE']

# Extract dish configuration from YAML
DISH_CONFIG = [(dish['dish'], dish['default_state'], dish['has_explicit_vegan_rule']) 
               for dish in remapping['dish_config']]

# Generate modification rules programmatically (keep the automation)
modification_rules_step1 = []
modification_rules_step2 = []

for base_name, final_default_state, has_explicit_vegan_rule in DISH_CONFIG:
    intermediate_name = f"Nonmeat {base_name}"
    meat_intermediate_name = f"Meat {base_name}"

    # Step 1 Rules (Base -> Intermediate)
    modification_rules_step1.extend([
        (base_name, MODS_VEGAN_MEAT, intermediate_name),
        (base_name, MODS_MEAT, meat_intermediate_name),
        (base_name, MODS_NONE, intermediate_name),
    ])

    # Step 2 Rules (Intermediate -> Final)
    vegan_final_name = f"Vegan {base_name}"
    vegetarian_final_name = f"Vegetarian {base_name}"
    meat_final_name = f"Meat {base_name}"

    if final_default_state == 'Vegan':
        default_final_name = vegan_final_name
    elif final_default_state == 'Vegetarian':
        default_final_name = vegetarian_final_name
    else:
        default_final_name = meat_final_name

    step2_rules = [
        (intermediate_name, MODS_VEGAN_DAIRY_EGG, vegan_final_name),
        (intermediate_name, MODS_DAIRY_EGG, vegetarian_final_name),
    ]

    if has_explicit_vegan_rule:
        step2_rules.append((intermediate_name, MODS_EXPLICIT_VEGAN, vegan_final_name))

    step2_rules.append((intermediate_name, MODS_NONE, default_final_name))
    modification_rules_step2.extend(step2_rules)

# Calculate rare items dynamically (items that appear less than 10 times)
rare_items = df_uncleaned['item_name'].value_counts().to_frame('counts').query('counts < 10').index.tolist()
drink_categories = remapping['drink_categories']
drink_items = df_uncleaned.query('dish_category.isin(@drink_categories)')['item_name'].value_counts().index.tolist()
items_to_remove = remapping['merch_list'] + drink_items + rare_items

df_uncleaned = remove_numbers(df_uncleaned, 'item_name')

# Step 1: Apply initial modifications
df_intermediate = fully_relabel_and_consolidate(
    df_uncleaned,
    remove=items_to_remove,
    name_changes=remapping.get("name_changes", {}),
    modification_name_changes=modification_rules_step1,
    vegan_list=remapping.get("vegan_list", []),
    vegetarian_list=remapping.get("vegetarian_list", []),
    meat_list=remapping.get("meat_list", []),
    alcohol_list=remapping.get("alcoholic_drinks", []),
    drinks_list=remapping.get("non_alcoholic_drinks", []),
    merch=remapping.get("merch_list", []),
    rare=rare_items + remapping.get("rare_list", []),
    unknown=remapping.get("unknown_list", []),
    remove_categories=drink_categories
)

# Step 2: Apply final modifications
df_relabeled = fully_relabel_and_consolidate(
    df_intermediate,
    modification_name_changes=modification_rules_step2,
    vegan_list=remapping.get("vegan_list", []),
    vegetarian_list=remapping.get("vegetarian_list", []),
    meat_list=remapping.get("meat_list", []),
    alcohol_list=remapping.get("alcoholic_drinks", []),
    drinks_list=remapping.get("non_alcoholic_drinks", []),
    merch=remapping.get("merch_list", []),
    rare=rare_items + remapping.get("rare_list", []),
    unknown=remapping.get("unknown_list", [])
)

df_relabeled.to_parquet(DATA_DIR_3_1 / (loc_id + '_sales_and_menu.parquet'))

# Consolidate dishes
consolidating_names = {dish: [f"Vegan {dish}", f"Vegetarian {dish}", f"Meat {dish}"] for dish, _, _ in DISH_CONFIG}
df_consolidated = df_relabeled.pipe(rename_items, name_changes=consolidating_names)

plot_dish_time_series(df_consolidated, loc_id, before_after_details_true, top_n=30)
df_consolidated.to_parquet(DATA_DIR_3_2 / (loc_id + '_sales_and_menu.parquet'))

In [ ]:
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
print(df_uncleaned.query('item_name.str.contains("Vegan") or item_modifications.str.contains("Vegan")').query('item_name.str.contains("Bacon") or item_modifications.str.contains("Bacon")')['item_quantity'].sum())
print(df_uncleaned.loc[promo_date:promo_date+pd.DateOffset(days=60)].query('item_name.str.contains("Vegan") or item_modifications.str.contains("Vegan")').query('item_name.str.contains("Bacon") or item_modifications.str.contains("Bacon")')['item_quantity'].sum())

# Parse promo strings / lists
promo_name = before_after_details_true.loc[loc_id,'promo_name']
if isinstance(promo_name, str) and promo_name.startswith('['):
    import ast
    promo_list = ast.literal_eval(promo_name)
    vegan_str, bacon_str = promo_list[0], promo_list[1]
else:
    vegan_str, bacon_str = "Vegan", "Bacon"

# Define filters
filters = {
    'Vegan Bacon': lambda df: (df['item_name'].str.contains(vegan_str, na=False) & df['item_name'].str.contains(bacon_str, na=False)) | (df['item_modifications'].str.contains(vegan_str, na=False) & df['item_modifications'].str.contains(bacon_str, na=False)),
    'Make It Vegan': lambda df: df['item_modifications'].str.contains("Make It Vegan", na=False),
    'Vegan Egg': lambda df: df['item_modifications'].str.contains("Vegan Egg|Just Egg", na=False),
    'Vegan Cream Cheese': lambda df: df['item_modifications'].str.contains("Vegan Cream Cheese", na=False),
    'Vegan Sausage': lambda df: df['item_modifications'].str.contains("Vegan Sausage", na=False),
    'Vegan Cheese': lambda df: df['item_modifications'].str.contains("Vegan Cheese|Vegan Cheddar", na=False)
}

# Plot results
plt.figure(figsize=(12, 6))
for label, filter_func in filters.items():
    data = df_uncleaned.loc[filter_func(df_uncleaned)]
    plt.plot(data.resample('W')['item_quantity'].sum(), label=label)
plt.axvline(x=promo_date, color='red', linestyle='--', label='Promo Date')
plt.title("Plant-Based Analog")
plt.legend()
plt.xticks(rotation=50)
plt.show()

### Dish Consolidation

In [ ]:
dish_conditions_egg_meat = [df['item_name'].str.contains(dish) for dish in ['Egg Meat', 'Egg Meat Cheese', 'Ex Meat']]
modification_meat_conditions = [df['item_modifications'].str.contains(meat) for meat in ['Sausage', 'Bacon', 'Ham']]
print(df
      .loc[reduce(np.logical_or, dish_conditions_egg_meat) & ~reduce(np.logical_or, modification_meat_conditions)]
      .value_counts(['item_name','item_modifications'], sort=False)
      .to_string())